# 00 Export Metadata: item titles + images for the app MVP

Produces `DATA_DIR/artifacts/items_metadata.parquet` (`item_id, domain, title,
image_url`) for the items already known to the trained SVD/ALS/item2vec/SASRec/BERT4Rec
models (Task 2 export cells in the `svd`/`als`/`item-2-vec`/`transformer` branches
must run first -- this notebook reads their `item2idx.json` / `word2vec.model`).

Colab-only: not executed as part of this change. `images` schema on the
McAuley-Lab/Amazon-Reviews-2023 HF dataset is not verified here -- see the
warning before `extract_image_url`.

## Setup

In [9]:
!test -d /content/recommendation-system/.git \
  || git clone -b app https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin app
!git checkout app
!git reset --hard origin/app
!git log -1 --oneline
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')


/content/recommendation-system
From https://github.com/mkobycheva/recommendation-system
 * branch            app        -> FETCH_HEAD
Already on 'app'
Your branch is up to date with 'origin/app'.
HEAD is now at 8761d19 Fix HF datasets loading: read from refs/convert/parquet, not the script
8761d19 (HEAD -> app, origin/app) Fix HF datasets loading: read from refs/convert/parquet, not the script
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports

In [10]:
import json
import os

import pandas as pd
from gensim.models import Word2Vec


## Load known items from each trained model

`item2vec` doesn't export its own `item2idx.json` (Task 2) -- its
`word2vec.model` is self-describing, `model.wv.index_to_key` *is* its
vocabulary, so we read that directly instead. `svd`/`als`/`sasrec`/`bert4rec`
all export `item2idx.json` with the same `add_domain_item_ids` convention
(`{domain}::{parent_asin}`).

In [11]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'
ARTIFACTS_DIR = f'{DATA_DIR}/artifacts'


def load_known_items(model_name):
    if model_name == 'item2vec':
        model = Word2Vec.load(f'{ARTIFACTS_DIR}/item2vec/word2vec.model')
        return set(model.wv.index_to_key)

    with open(f'{ARTIFACTS_DIR}/{model_name}/item2idx.json') as f:
        return set(json.load(f).keys())


# Union, not intersection of the five models with each other: we want any
# item that at least one model knows, so it's not "no model knows this title".
known_items = set()
for model_name in ('svd', 'als', 'item2vec', 'sasrec', 'bert4rec'):
    known_items |= load_known_items(model_name)

needed_asins = {
    'books': {item_id.split('::', 1)[1] for item_id in known_items if item_id.startswith('books::')},
    'movies': {item_id.split('::', 1)[1] for item_id in known_items if item_id.startswith('movies::')},
}
print({domain: len(asins) for domain, asins in needed_asins.items()})


{'books': 404630, 'movies': 182434}


## Stream-filter the Amazon Reviews 2023 metadata catalog

`meta_Books.jsonl.gz`/`meta_Movies_and_TV.jsonl.gz` are the full catalog
(millions of rows, gzipped -- Books alone is ~4.9GB compressed) -- we only
need the `needed_asins` intersection (thousands of items).

**Not using the HF `datasets` library here**: `datasets>=4.0` removed
loading-script support entirely, and this repo's `raw_meta_Books`/
`raw_meta_Movies_and_TV` configs still rely on `Amazon-Reviews-2023.py`
(only 9 smaller categories were auto-converted to the parquet revision --
Books/Movies_and_TV aren't among them, confirmed against the repo's file
listing). Instead, stream the raw `.jsonl.gz` directly over HTTP -- same
pattern `src/data/overlap_check.py` already uses for the review files,
just for `raw/meta_categories/` instead of `raw/review_categories/`.
`gzip.open` on the HTTP response decompresses line-by-line as it downloads;
nothing is cached to disk and non-matches are discarded immediately.


In [ ]:
import gzip
import ssl
from urllib.request import urlopen

BASE_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories'
CATEGORY_FILENAMES = {
    'books': 'meta_Books.jsonl.gz',
    'movies': 'meta_Movies_and_TV.jsonl.gz',
}


def open_public_dataset_url(url):
    """Same approach as src/data/overlap_check.py -- these files are public
    and immutable, an unverified SSL context keeps this runnable without
    extra cert setup."""
    context = ssl._create_unverified_context()
    return urlopen(url, context=context)


def collect_metadata(filename, needed, progress_every=200_000):
    url = f'{BASE_URL}/{filename}'
    matched = []
    with open_public_dataset_url(url) as response:
        with gzip.open(response, 'rt', encoding='utf-8') as f:
            for i, line in enumerate(f, start=1):
                row = json.loads(line)
                if row.get('parent_asin') in needed:
                    matched.append(row)
                    if len(matched) == len(needed):  # early stop once everything is found
                        break
                if i % progress_every == 0:
                    print(f'  ...scanned {i:,} rows, matched {len(matched):,}/{len(needed):,}')
    return matched


matched_rows = {}
for domain, filename in CATEGORY_FILENAMES.items():
    print(f'Streaming {filename}, looking for {len(needed_asins[domain]):,} items...')
    matched_rows[domain] = collect_metadata(filename, needed_asins[domain])
    print(f'  found {len(matched_rows[domain]):,} / {len(needed_asins[domain]):,}')


## Save the filtered raw rows locally (separate from the final parquet)

The expensive pass (streaming millions of rows) happens once here. Parsing
`title`/`images` below will likely need a few iterations to match the real
schema -- that parsing step reads this small local file, not the network.

In [ ]:
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

for domain, rows in matched_rows.items():
    path = f'{ARTIFACTS_DIR}/matched_{domain}_raw.jsonl'
    with open(path, 'w') as f:
        for row in rows:
            f.write(json.dumps(row, default=str) + '\n')
    print(f'Saved {len(rows):,} rows to {path}')


### Reuse across reruns (not used here)

If this ever needs to be rerun for a different `needed_asins` and Colab
disk space allows, downloading the `.jsonl.gz` once to Drive and streaming
from the local copy avoids re-fetching ~5GB from the network on every rerun:

```python
!wget -q -O /content/drive/MyDrive/recsys-data/meta_Books.jsonl.gz \
    https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Books.jsonl.gz
```

then `gzip.open('/content/drive/MyDrive/recsys-data/meta_Books.jsonl.gz', 'rt')`
instead of streaming the URL directly. Not used above since this notebook
only needs to run once per model refresh, and the local copy would count
against Drive storage.


## Parse title + image

⚠️ **Schema not verified without running this notebook.** `raw_meta_*` on HF
typically has `parent_asin`, `title`, `images`, among others, but the exact
shape of `images` (dict-of-lists from Arrow conversion vs. list-of-dict from
raw jsonl vs. plain URL list/string) needs to be checked against
`matched_rows['books'][0]` the first time this actually runs, and
`extract_image_url` adjusted if none of the branches below match.

In [ ]:
def extract_image_url(row):
    images = row.get('images')
    if not images:
        return None

    # Most likely on HF: dict-of-lists (Arrow-friendly columnar form), e.g.
    # {"hi_res": [...], "large": [...], "thumb": [...], "variant": [...]}.
    if isinstance(images, dict):
        for key in ('hi_res', 'large', 'thumb'):
            values = images.get(key)
            if values:
                first = next((v for v in values if v), None)
                if first:
                    return first

    # Fallback: raw-jsonl style list of per-image dicts.
    if isinstance(images, list) and images and isinstance(images[0], dict):
        for item in images:
            for key in ('hi_res', 'large', 'thumb'):
                if item.get(key):
                    return item[key]

    # Fallback: plain list of URL strings, or a single URL string.
    if isinstance(images, list) and images and isinstance(images[0], str):
        return images[0]
    if isinstance(images, str):
        return images

    return None


In [ ]:
def parse_rows(rows, domain):
    parsed = []
    for row in rows:
        parent_asin = row.get('parent_asin')
        if not parent_asin:
            continue
        parsed.append({
            'item_id': f'{domain}::{parent_asin}',
            'domain': domain,
            'title': row.get('title') or '',
            'image_url': extract_image_url(row),
        })
    return parsed


items_metadata_rows = []
for domain, rows in matched_rows.items():
    items_metadata_rows.extend(parse_rows(rows, domain))

items_metadata = pd.DataFrame(items_metadata_rows, columns=['item_id', 'domain', 'title', 'image_url'])
items_metadata.head()


## Save `items_metadata.parquet`

In [ ]:
output_path = f'{ARTIFACTS_DIR}/items_metadata.parquet'
items_metadata.to_parquet(output_path, index=False)
print(f'Saved {len(items_metadata):,} rows to {output_path}')


## Publish artifacts for the deployed app (HF Spaces)

The Space that serves this app can't hold the ~4GB `artifacts/` folder in
its own git repo (Spaces cap that at 1GB on the free tier) -- it downloads
them at startup from a companion **public HF Dataset repo** instead
(`huggingface_hub.snapshot_download`, no token needed for a public repo).
This cell publishes `ARTIFACTS_DIR` there. Run it after every model
refresh so the deployed Space picks up the latest artifacts on its next
restart.

In [ ]:
from huggingface_hub import HfApi, notebook_login

notebook_login()  # skip if already authenticated in this session

ARTIFACTS_REPO = '<your-hf-username>/recsys-artifacts'  # public dataset repo

api = HfApi()
api.create_repo(ARTIFACTS_REPO, repo_type='dataset', exist_ok=True, private=False)
api.upload_folder(folder_path=ARTIFACTS_DIR, repo_id=ARTIFACTS_REPO, repo_type='dataset')
print(f'Uploaded {ARTIFACTS_DIR} to https://huggingface.co/datasets/{ARTIFACTS_REPO}')
